# Data audit

This notebook inspects raw schemas, date coverage, missingness, and the conservative fundamental availability lag. It is diagnostic only; production transformations live in `src/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import DataCatalog
from src.utils.config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
catalog = DataCatalog(PROJECT_ROOT / config['paths']['raw_data_dir'], config['data'])

## Price coverage and duplicate keys

In [ ]:
prices = catalog.load_prices()
{
    'rows': len(prices),
    'symbols': prices['symbol'].nunique(),
    'start': prices['date'].min(),
    'end': prices['date'].max(),
    'duplicate_symbol_dates': prices.duplicated(['symbol', 'date']).sum(),
}

## Fundamental availability proxy

The supplied file has fiscal period end but no release timestamp. The production loader therefore applies the configured conservative lag and preserves both dates.

In [ ]:
fundamentals = catalog.load_fundamentals()
fundamentals[['symbol', 'period_ending', 'fundamental_available_date']].head()

## News timestamp quality

In [ ]:
news = catalog.load_news(
    symbols=prices['symbol'].unique(),
    start=config['universe']['start_date'],
    end='2017-01-01',
)
{
    'rows': len(news),
    'symbols': news['symbol'].nunique(),
    'start': news['published_at'].min(),
    'end': news['published_at'].max(),
    'missing_headline_share': news['headline'].isna().mean(),
}

## Required audit notes

Record any universe gaps, timestamp anomalies, sparse fields, and assumptions here before running the search. Do not inspect 2016 factor performance in this notebook.